# mechanism_viewer examples

## Use accuracy mar for detection of missing data mechanisms
This notebook contains examples in how to use functions belonging to `accuracy_mar.py`, which are related to identifying the MAR missing data mechanism in one column of a dataset.

By training a classification model to predict whether a value in a given column is missing, using only the fully observed columns as predictors, and then evaluating the model’s accuracy, we can assess the likely missing‑data mechanism for that column. If the model achieves high predictive accuracy, this suggests that the missingness can be explained by the observed variables, indicating that the underlying mechanism is likely MAR.

Furthermore, for reproducible experiments, the functions can use `DEFAULT_RANDOM_STATE`. Otherwise, the user can change the seed in the parameter `random_state`.

### 1. Generate synthetic datasets with missing values

First, let's generate a synthetic dataset with some missing values. The dataset that will be generated will have two complete columns, Col1 and Col2, and three coluns with different missing data mechanisms each: Col3 (MAR), Col4 (MCAR) and Col5 (MNAR).

In [2]:
from mechanism_viewer import generate_dataset_with_missing_data, ColType

data = generate_dataset_with_missing_data(100, [ColType.DISCRETE, ColType.CONTINUOUS ,ColType.CONTINUOUS,
                                                 ColType.CONTINUOUS, ColType.CONTINUOUS], 2, ["MAR", "MCAR", "MNAR"], [0.2, 0.2, 0.2])

display(data.head(10))

,Col1,Col2,Col3,Col4,Col5
0,5,-0.062679,0.086590,1.848956,-0.923233
1,4,0.955142,-0.155677,NaN,-1.351685
2,4,-0.985726,1.167782,-0.268889,-0.975873
3,5,0.504047,0.254421,NaN,NaN
4,5,-0.530258,0.337603,2.573360,-0.949399
5,3,-0.792873,-0.411877,0.059218,NaN
6,5,-0.107030,-0.487606,NaN,0.493318
7,4,-1.035242,-0.432558,-0.024125,0.184836
8,6,-0.553649,0.394452,0.198085,-0.858358
9,7,-1.197878,-0.420984,NaN,0.700310


### 2. Identifying MAR mechanism with `run_random_forest()`

As the name suggests, the `run_random_forest()` function trains a random forest model to determine how well the missingness of a column can be predicted from the observed columns.

The next cell executes `run_random_forest()` for each column with missing values (Col3, Col4, and Col5), using the fully observed columns as predictors (Col1 and Col2). This produces an accuracy score for every model trained on columns with different missing‑data mechanisms. By comparing these accuracies, we can see how each mechanism behaves. In particular, the model trained on the MAR column will typically achieve higher accuracy than those trained on the MCAR or MNAR columns.

In [3]:
from mechanism_viewer import run_random_forest

acc_3_rf = run_random_forest(data, "Col3")    # Col3 has MAR
acc_4_rf = run_random_forest(data, "Col4")    # Col4 has MCAR
acc_5_rf = run_random_forest(data, "Col5")    # Col5 has MNAR

Random Forest Model Accuracy: 0.9333
Random Forest Model Accuracy: 0.8333
Random Forest Model Accuracy: 0.7333


> Note: The resulting accuracy is returned as float.

### 3. Identifying MAR mechanism with `run_logistic_regression()`

The function `run_logistic_regression()` can also be used to detect MAR missing mechanism using the same parameters.


In [4]:
from mechanism_viewer import run_logistic_regression

acc_3_lr = run_logistic_regression(data, "Col3")   # Col3 has MAR
acc_4_lr = run_logistic_regression(data, "Col4")   # Col4 has MCAR
acc_5_lr = run_logistic_regression(data, "Col5")   # Col5 has MNAR

Logistic Regression Model Accuracy: 1.0000
Logistic Regression Model Accuracy: 0.8000
Logistic Regression Model Accuracy: 0.8000


### 4. Identifying MAR mechanism with `detect_mar_from_model_accuracy()`

Detecting MAR with `run_random_forest()` or `run_logistic_regression()` alone can be quite challenging, since different missing data mechanisms often produce similarly high accuracy values. However, the `detect_mar_from_model_accuracy()` function addresses this by combining the accuracies from both models and comparing the resulting score against a baseline based on the missing rate of the missing column. This score is called Accuracy Baseline Difference (ABD) and it allows us to better distinguish MAR from MCAR and MNAR.

It is important to note that `detect_mar_from_model_accuracy()` can print the most likely missing data mechanism of the missing column, with its parameter `print_result` set to `True`, which is its default value.

Moreover, the interpretation rule of `detect_mar_from_model_accuracy()` is configurable through the parameter `threshold`, being its default value `5.0`.

Finally, the function `interpret_accuracy_baseline_diff()` can be called directly, to only obtain the interpretation text, using an obtained ABD.

In [5]:
from mechanism_viewer import detect_mar_from_model_accuracy, interpret_accuracy_baseline_diff

# Using the compound function that calculates the difference of model accuracy, the Accuracy Baseline Difference (ABD).
acc_3 = detect_mar_from_model_accuracy(data, "Col3")   # Col3 has MAR
print()
acc_4 = detect_mar_from_model_accuracy(data, "Col4")   # Col4 has MCAR
print()
acc_5 = detect_mar_from_model_accuracy(data, "Col5")   # Col5 has MNAR

# Since the ABD values are already obtained, they can be interpreted using interpret_accuracy_baseline_diff()
print()
print("Col3:", interpret_accuracy_baseline_diff(acc_3))
print("Col4:", interpret_accuracy_baseline_diff(acc_4))
print("Col5:", interpret_accuracy_baseline_diff(acc_5))

Random Forest Model Accuracy: 0.9333
Logistic Regression Model Accuracy: 1.0000
The target column Col3 with missing rate of 0.2 gives an Accuracy Baseline Difference of 16.67.
Since the obtained Accuracy Baseline Difference is 16.67, and the given threshold is 5.0, thus, it is likely that the underlying mechanism of the missing column is MAR.

Random Forest Model Accuracy: 0.8333
Logistic Regression Model Accuracy: 0.8000
The target column Col4 with missing rate of 0.2 gives an Accuracy Baseline Difference of 1.67.
Since the obtained Accuracy Baseline Difference is 1.67, and the given threshold is 5.0, thus, it is likely that the underlying mechanism of the missing column is MCAR/MNAR.

Random Forest Model Accuracy: 0.7333
Logistic Regression Model Accuracy: 0.8000
The target column Col5 with missing rate of 0.2 gives an Accuracy Baseline Difference of -3.33.
Since the obtained Accuracy Baseline Difference is -3.33, and the given threshold is 5.0, thus, it is likely that the underlying

> Note: To not print any commentary in `run_random_forest()`, in `run_logistic_regression()` or in `detect_mar_from_model_accuracy()`, use `print_result = False` as the last parameter.

### 5. Custom interpretation threshold and silent mode

Use `print_result=False` to suppress the function prints and control output manually.

Futhermore, different thresholds can be used in `interpret_accuracy_baseline_diff()`, by setting the wished value in `threshold` parameter.

In [10]:
custom_threshold = 3.0

acc_3_silent = detect_mar_from_model_accuracy(data, "Col3", threshold=custom_threshold, print_result=False)

interpretation_3 = interpret_accuracy_baseline_diff(acc_3_silent, threshold=custom_threshold)

print(interpretation_3)

Since the obtained Accuracy Baseline Difference is 16.67, and the given threshold is 3.0, thus, it is likely that the underlying mechanism of the missing column is MAR.
